In [1]:
!pip install lightgbm

In [2]:
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import StratifiedKFold
# from sklearn.preprocessing import StandardScaler
# from sklearn.naive_bayes import GaussianNB
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import SVC
# from sklearn.decomposition import PCA
# from sklearn.cross_decomposition import PLSRegression
# from sklearn.metrics import roc_curve, auc, accuracy_score, f1_score, precision_score, recall_score
# from xgboost import XGBClassifier
# from lightgbm import LGBMClassifier


# # ====================== READ CSV ======================

# data = pd.read_csv('/kaggle/input/competitions/santander-customer-transaction-prediction/train.csv')
# test_data = pd.read_csv('/kaggle/input/competitions/santander-customer-transaction-prediction/test.csv')

# print(f"Train: {data.shape}   Test: {test_data.shape}")
# print(f"\nTarget distribution:\n{data['target'].value_counts()}")
# print(f"\nColumns: {list(data.columns[:5])} ... {list(data.columns[-3:])}")


# # ====================== PREPARE DATA ======================

# X_all = data.drop(columns=['target', 'ID_code']).values
# y_all = data['target'].values
# X_test = test_data.drop(columns=['ID_code']).values
# test_ids = test_data['ID_code'].values

# print(f"\nX_all: {X_all.shape}   X_test: {X_test.shape}")


# # ====================== HELPERS ======================

# def best_threshold(y_true, scores):
#     """Threshold maximizing Youden's J = TPR - FPR."""
#     fpr, tpr, thr_arr = roc_curve(y_true, scores)
#     J = tpr - fpr
#     return thr_arr[int(np.argmax(J))]


# def make_submission(model_name, y_true, train_scores, test_scores, test_ids):
#     """Find best threshold, predict on test, save submission."""
#     thr = best_threshold(y_true, train_scores)
#     preds = (test_scores >= thr).astype(int)

#     # metrics on train (OOF if CV, in-sample if not)
#     fpr, tpr, _ = roc_curve(y_true, train_scores)
#     auc_prob = auc(fpr, tpr)
#     J = (tpr - fpr).max()
#     auc_bin = (1 + J) / 2

#     sub = pd.DataFrame({'ID_code': test_ids, 'target': preds})
#     fname = f'submission_{model_name}.csv'
#     sub.to_csv(fname, index=False)

#     print(f"  {model_name:25s}  AUC_prob={auc_prob:.4f}  AUC_bin={auc_bin:.4f}  "
#           f"thr={thr:+.3f}  pred_pos={preds.mean()*100:.2f}%  -> {fname}")

#     return {
#         'model': model_name, 'auc_prob': auc_prob, 'auc_binary': auc_bin,
#         'threshold': thr, 'pred_pos_pct': preds.mean() * 100
#     }


# # ====================== TRAIN + SUBMIT EACH MODEL ======================

# results = []

# # scale
# scaler = StandardScaler()
# X_sc = scaler.fit_transform(X_all)
# X_test_sc = scaler.transform(X_test)

# # ---- 1. GNB ----
# print("\n--- GNB ---")
# m = GaussianNB(var_smoothing=1e-6)
# m.fit(X_sc, y_all)
# train_scores = m.predict_proba(X_sc)[:, 1]
# test_scores = m.predict_proba(X_test_sc)[:, 1]
# results.append(make_submission('gnb', y_all, train_scores, test_scores, test_ids))


# # ---- 2. KNN LDA-1 (k=301, uniform) ----
# print("\n--- KNN LDA-1 ---")
# lda_red = LinearDiscriminantAnalysis(solver='svd')
# X_lda = lda_red.fit_transform(X_sc, y_all)
# X_test_lda = lda_red.transform(X_test_sc)

# m = KNeighborsClassifier(n_neighbors=301, weights='uniform', n_jobs=-1)
# m.fit(X_lda, y_all)
# train_scores = m.predict_proba(X_lda)[:, 1]
# test_scores = m.predict_proba(X_test_lda)[:, 1]
# results.append(make_submission('knn_lda1', y_all, train_scores, test_scores, test_ids))


# # ---- 3. KNN PLS-DA-30 (k=301, distance) ----
# print("\n--- KNN PLS-DA-30 ---")
# pls = PLSRegression(n_components=30, scale=False)
# X_pls = pls.fit_transform(X_sc, y_all.reshape(-1, 1))[0]
# X_test_pls = pls.transform(X_test_sc)

# m = KNeighborsClassifier(n_neighbors=301, weights='distance', n_jobs=-1)
# m.fit(X_pls, y_all)
# train_scores = m.predict_proba(X_pls)[:, 1]
# test_scores = m.predict_proba(X_test_pls)[:, 1]
# results.append(make_submission('knn_pls30', y_all, train_scores, test_scores, test_ids))


# # ---- 4. KNN PCA-30 (k=301, distance) ----
# print("\n--- KNN PCA-30 ---")
# pca = PCA(n_components=30, random_state=42)
# X_pca = pca.fit_transform(X_sc)
# X_test_pca = pca.transform(X_test_sc)

# m = KNeighborsClassifier(n_neighbors=301, weights='distance', n_jobs=-1)
# m.fit(X_pca, y_all)
# train_scores = m.predict_proba(X_pca)[:, 1]
# test_scores = m.predict_proba(X_test_pca)[:, 1]
# results.append(make_submission('knn_pca30', y_all, train_scores, test_scores, test_ids))



# # ---- 6. ElasticNet LR (C=2.79, l1_ratio=0.5) ----
# print("\n--- ElasticNet LR ---")
# m = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5,
#                        C=2.7941, max_iter=2000, tol=1e-3, random_state=42)
# m.fit(X_sc, y_all)
# train_scores = m.predict_proba(X_sc)[:, 1]
# test_scores = m.predict_proba(X_test_sc)[:, 1]
# results.append(make_submission('elasticnet_lr', y_all, train_scores, test_scores, test_ids))


# # ---- 7. LDA classifier (shrinkage=auto) ----
# print("\n--- LDA classifier ---")
# m = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
# m.fit(X_sc, y_all)
# train_scores = m.predict_proba(X_sc)[:, 1]
# test_scores = m.predict_proba(X_test_sc)[:, 1]
# results.append(make_submission('lda', y_all, train_scores, test_scores, test_ids))


# # ---- 8. XGBoost ----
# print("\n--- XGBoost ---")
# m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
#                   subsample=0.8, colsample_bytree=0.8,
#                   tree_method='hist', eval_metric='auc',
#                   n_jobs=-1, random_state=42)
# m.fit(X_sc, y_all)
# train_scores = m.predict_proba(X_sc)[:, 1]
# test_scores = m.predict_proba(X_test_sc)[:, 1]
# results.append(make_submission('xgboost', y_all, train_scores, test_scores, test_ids))


# # ---- 9. LightGBM ----
# print("\n--- LightGBM ---")
# m = LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
#                    subsample=0.8, colsample_bytree=0.8,
#                    is_unbalance=True,
#                    n_jobs=-1, random_state=42, verbose=-1)
# m.fit(X_sc, y_all)
# train_scores = m.predict_proba(X_sc)[:, 1]
# test_scores = m.predict_proba(X_test_sc)[:, 1]
# results.append(make_submission('lightgbm', y_all, train_scores, test_scores, test_ids))


# # ====================== SUMMARY ======================

# df_res = pd.DataFrame(results).sort_values('auc_binary', ascending=False)

# print("\n" + "=" * 95)
# print("ALL 8 MODELS — individual submissions (ranked by binary AUC)")
# print("=" * 95)
# print(df_res.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# print("\n" + "=" * 95)
# print("SUBMISSION FILES READY:")
# print("=" * 95)
# for _, row in df_res.iterrows():
#     print(f"  submission_{row['model']}.csv  (AUC_bin={row['auc_binary']:.4f})")

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import roc_curve, auc, accuracy_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# ====================== READ CSV ======================

data = pd.read_csv('/kaggle/input/competitions/santander-customer-transaction-prediction/train.csv')
test_data = pd.read_csv('/kaggle/input/competitions/santander-customer-transaction-prediction/test.csv')

print(f"Train: {data.shape}   Test: {test_data.shape}")


# ====================== PREPARE DATA ======================

X_all = data.drop(columns=['target', 'ID_code']).values
y_all = data['target'].values
X_test = test_data.drop(columns=['ID_code']).values
test_ids = test_data['ID_code'].values

print(f"X_all: {X_all.shape}   X_test: {X_test.shape}")


# ====================== HELPER: TRAIN 9 MODELS AND PREDICT ======================

def train_and_predict_8(X_tr, y_tr, X_pred):
    """Train all 8 models on (X_tr, y_tr), predict on X_pred.
    Returns array of shape (len(X_pred), 8) with probability scores."""

    scores = np.zeros((len(X_pred), 9))

    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_pred_sc = sc.transform(X_pred)

    # ---- 1. GNB ----
    m = GaussianNB(var_smoothing=1e-6)
    m.fit(X_tr_sc, y_tr)
    scores[:, 0] = m.predict_proba(X_pred_sc)[:, 1]

    # ---- 2. KNN LDA-1 (k=301, uniform) ----
    lda_red = LinearDiscriminantAnalysis(solver='svd')
    X_tr_lda = lda_red.fit_transform(X_tr_sc, y_tr)
    X_pred_lda = lda_red.transform(X_pred_sc)

    m = KNeighborsClassifier(n_neighbors=301, weights='uniform', n_jobs=-1)
    m.fit(X_tr_lda, y_tr)
    scores[:, 1] = m.predict_proba(X_pred_lda)[:, 1]

    # ---- 3. KNN PLS-DA-30 (k=301, distance) ----
    pls = PLSRegression(n_components=30, scale=False)
    X_tr_pls = pls.fit_transform(X_tr_sc, y_tr.reshape(-1, 1))[0]
    X_pred_pls = pls.transform(X_pred_sc)

    m = KNeighborsClassifier(n_neighbors=301, weights='distance', n_jobs=-1)
    m.fit(X_tr_pls, y_tr)
    scores[:, 2] = m.predict_proba(X_pred_pls)[:, 1]

    # ---- 4. KNN PCA-30 (k=301, distance) ----
    pca = PCA(n_components=30, random_state=42)
    X_tr_pca = pca.fit_transform(X_tr_sc)
    X_pred_pca = pca.transform(X_pred_sc)

    m = KNeighborsClassifier(n_neighbors=301, weights='distance', n_jobs=-1)
    m.fit(X_tr_pca, y_tr)
    scores[:, 3] = m.predict_proba(X_pred_pca)[:, 1]

    # ---- 6. ElasticNet LR (C=2.79, l1_ratio=0.5) ----
    m = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5,
                           C=2.7941, max_iter=2000, tol=1e-3, random_state=42)
    m.fit(X_tr_sc, y_tr)
    scores[:, 4] = m.predict_proba(X_pred_sc)[:, 1]

    # ---- 7. LDA classifier (shrinkage=auto) ----
    m = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
    m.fit(X_tr_sc, y_tr)
    scores[:, 5] = m.predict_proba(X_pred_sc)[:, 1]

    # ---- 8. XGBoost ----
    m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                      subsample=0.8, colsample_bytree=0.8,
                      tree_method='hist', eval_metric='auc',
                      n_jobs=-1, random_state=42)
    m.fit(X_tr_sc, y_tr)
    scores[:, 6] = m.predict_proba(X_pred_sc)[:, 1]

    # ---- 9. LightGBM ----
    m = LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                       subsample=0.8, colsample_bytree=0.8,
                       is_unbalance=True,
                       n_jobs=-1, random_state=42, verbose=-1)
    m.fit(X_tr_sc, y_tr)
    scores[:, 7] = m.predict_proba(X_pred_sc)[:, 1]

    return scores


MODEL_NAMES = ['GNB', 'KNN_LDA1', 'KNN_PLS30', 'KNN_PCA30', 'ElasticNet_LR', 'LDA', 'XGBoost', 'LightGBM']


# ====================== STEP 1: SPLIT TRAIN (stratified) ======================

print("\n===== STEP 1: Split train data =====")

X_meta_train, X_meta_val, y_meta_train, y_meta_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

print(f"  Meta-train:    {X_meta_train.shape}  (80%)")
print(f"  Meta-val:      {X_meta_val.shape}  (20%)")


# ====================== STEP 2: TRAIN 9 MODELS ON META-TRAIN ======================

print("\n===== STEP 2: Train 8 models on meta-train =====")

meta_val_scores = train_and_predict_8(X_meta_train, y_meta_train, X_meta_val)

print(f"  Meta-val scores shape: {meta_val_scores.shape}  (rows x 8 models)")


# ====================== STEP 3: PREDICT ON META-VAL (already done above) ======================

print("\n===== STEP 3: Meta-validation scores ready =====")

for i, name in enumerate(MODEL_NAMES):
    fpr, tpr, _ = roc_curve(y_meta_val, meta_val_scores[:, i])
    auc_i = auc(fpr, tpr)
    print(f"  {name:15s}  AUC_prob={auc_i:.4f}")


# ====================== STEP 4: TRAIN LOGISTIC REGRESSION (THE WEIGHTS) ======================

print("\n===== STEP 4: Train LR to learn weights =====")

meta_lr = LogisticRegression(max_iter=1000, random_state=42)
meta_lr.fit(meta_val_scores, y_meta_val)

# show the learned weights
print(f"\n  LR intercept: {meta_lr.intercept_[0]:+.4f}")
print(f"  Learned weights (coefficients):")
for name, w in sorted(zip(MODEL_NAMES, meta_lr.coef_[0]), key=lambda x: -abs(x[1])):
    print(f"    {name:15s}  weight={w:+.4f}")


# ====================== STEP 5: FIND BEST THRESHOLD ======================

print("\n===== STEP 5: Find best threshold =====")

meta_val_probs = meta_lr.predict_proba(meta_val_scores)[:, 1]

fpr, tpr, thr_arr = roc_curve(y_meta_val, meta_val_probs)
J = tpr - fpr
best_idx = int(np.argmax(J))
best_thr = thr_arr[best_idx]

auc_prob_meta = auc(fpr, tpr)
auc_bin_meta = (1 + J[best_idx]) / 2

print(f"  Best threshold  = {best_thr:.4f}")
print(f"  AUC (prob)      = {auc_prob_meta:.4f}")
print(f"  AUC (binary)    = {auc_bin_meta:.4f}")
print(f"  TPR={tpr[best_idx]:.4f}  FPR={fpr[best_idx]:.4f}  J={J[best_idx]:.4f}")


# ====================== STEP 6: RETRAIN 9 MODELS ON FULL TRAIN ======================

print("\n===== STEP 6: Retrain all 8 models on FULL train data =====")

test_scores_full = train_and_predict_8(X_all, y_all, X_test)

print(f"  Test scores shape: {test_scores_full.shape}")


# ====================== STEP 7: FEED THROUGH LR → FINAL PROBABILITIES ======================

print("\n===== STEP 7: Feed through LR → final predictions =====")

test_probs = meta_lr.predict_proba(test_scores_full)[:, 1]
test_preds = (test_probs >= best_thr).astype(int)

print(f"  Predicted positives: {test_preds.sum()} ({test_preds.mean()*100:.2f}%)")


# ====================== STEP 8: WRITE SUBMISSION ======================

print("\n===== STEP 8: Write submission =====")

submission = pd.DataFrame({
    'ID_code': test_ids,
    'target': test_preds
})
submission.to_csv('submission_weighted_ensemble.csv', index=False)

print(f"  Saved: submission_weighted_ensemble.csv")
print(f"  Rows: {len(submission)}")
print(f"\n  Sample:")
print(submission.head(10).to_string(index=False))


# ====================== SUMMARY ======================

print("\n" + "=" * 85)
print("WEIGHTED ENSEMBLE SUMMARY")
print("=" * 85)
print(f"  Models combined : {len(MODEL_NAMES)}")
print(f"  Meta-train      : {X_meta_train.shape[0]} rows")
print(f"  Meta-val        : {X_meta_val.shape[0]} rows")
print(f"  Threshold       : {best_thr:.4f}")
print(f"  Meta-val AUC (prob)   : {auc_prob_meta:.4f}")
print(f"  Meta-val AUC (binary) : {auc_bin_meta:.4f}")
print(f"\n  Individual model AUCs (on meta-val):")
for i, name in enumerate(MODEL_NAMES):
    fpr_i, tpr_i, _ = roc_curve(y_meta_val, meta_val_scores[:, i])
    auc_i = auc(fpr_i, tpr_i)
    J_i = (tpr_i - fpr_i).max()
    auc_bin_i = (1 + J_i) / 2
    w = meta_lr.coef_[0][i]
    print(f"    {name:15s}  AUC_prob={auc_i:.4f}  AUC_bin={auc_bin_i:.4f}  weight={w:+.4f}")
print("=" * 85)

Train: (200000, 202)   Test: (200000, 201)
X_all: (200000, 200)   X_test: (200000, 200)

===== STEP 1: Split train data =====
  Meta-train:    (160000, 200)  (80%)
  Meta-val:      (40000, 200)  (20%)

===== STEP 2: Train 8 models on meta-train =====


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Meta-val scores shape: (40000, 9)  (rows x 8 models)

===== STEP 3: Meta-validation scores ready =====
  GNB              AUC_prob=0.8882
  KNN_LDA1         AUC_prob=0.8557
  KNN_PLS30        AUC_prob=0.8527
  KNN_PCA30        AUC_prob=0.8454
  ElasticNet_LR    AUC_prob=0.8599
  LDA              AUC_prob=0.8598
  XGBoost          AUC_prob=0.8801
  LightGBM         AUC_prob=0.8805

===== STEP 4: Train LR to learn weights =====

  LR intercept: -4.5854
  Learned weights (coefficients):
    LightGBM         weight=+3.8161
    GNB              weight=+2.8172
    KNN_PCA30        weight=+2.5393
    KNN_LDA1         weight=-1.8022
    KNN_PLS30        weight=+1.6787
    XGBoost          weight=+1.0392
    ElasticNet_LR    weight=+0.6348
    LDA              weight=-0.1103

===== STEP 5: Find best threshold =====
  Best threshold  = 0.0835
  AUC (prob)      = 0.8899
  AUC (binary)    = 0.8112
  TPR=0.8005  FPR=0.1781  J=0.6224

===== STEP 6: Retrain all 8 models on FULL train data =====


/usr/local/lib/python3.13/dist-packages/sklearn/cross_decomposition/_pls.py:365: RuntimeWarning: invalid value encountered in divide
  y_scores = np.dot(yk, y_weights) / y_ss
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Test scores shape: (200000, 9)

===== STEP 7: Feed through LR → final predictions =====
  Predicted positives: 50319 (25.16%)

===== STEP 8: Write submission =====
  Saved: submission_weighted_ensemble.csv
  Rows: 200000

  Sample:
ID_code  target
 test_0       1
 test_1       1
 test_2       1
 test_3       1
 test_4       0
 test_5       0
 test_6       0
 test_7       1
 test_8       0
 test_9       0

WEIGHTED ENSEMBLE SUMMARY
  Models combined : 8
  Meta-train      : 160000 rows
  Meta-val        : 40000 rows
  Threshold       : 0.0835
  Meta-val AUC (prob)   : 0.8899
  Meta-val AUC (binary) : 0.8112

  Individual model AUCs (on meta-val):
    GNB              AUC_prob=0.8882  AUC_bin=0.8070  weight=+2.8172
    KNN_LDA1         AUC_prob=0.8557  AUC_bin=0.7777  weight=-1.8022
    KNN_PLS30        AUC_prob=0.8527  AUC_bin=0.7733  weight=+1.6787
    KNN_PCA30        AUC_prob=0.8454  AUC_bin=0.7713  weight=+2.5393
    ElasticNet_LR    AUC_prob=0.8599  AUC_bin=0.7811  weight=+0.6348
